# 02 · Wizualizacja danych eye-trackingowych i analiza obszarów zainteresowania (AOI)

**Kurs:** Praktyczna Analiza Danych Eye-Tracking i EEG w Pythonie

W notebooku 01 zamieniliśmy surowe próbki na listę fiksacji. Teraz zajmiemy się pytaniem, **na co i jak długo patrzyli uczestnicy badania**:

1. ścieżki wzroku (*scanpath*) na tle bodźca – statyczne i interaktywne,
2. mapy cieplne (*heatmaps*) oparte na estymacji jądrowej gęstości (KDE),
3. analiza obszarów zainteresowania (AOI): czas do pierwszej fiksacji (*Time to First Fixation*, TTFF) i czas przebywania w obszarze (*Dwell Time*),
4. porównanie dwóch grup uczestników i prosty test statystyczny.

**Dane:** raport fiksacji 40 osób, które przez 10 s oglądały plakat reklamowy kawy. Połowa osób oglądała go swobodnie (`swobodne`), a połowa dostała zadanie: *„Oceń, czy produkt jest tani”* (`ocena_ceny`). To współczesna wersja klasycznego eksperymentu Yarbusa (1967), który pokazał, że **zadanie zmienia sposób oglądania** tego samego obrazu.

**Szacowany czas:** ok. 90 minut.

## 0. Przygotowanie środowiska

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from matplotlib.patches import Patch, Rectangle
from scipy import stats
from scipy.stats import gaussian_kde

KATALOG_GLOWNY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DANE = KATALOG_GLOWNY / "dane"
WYNIKI = KATALOG_GLOWNY / "wyniki"
WYNIKI.mkdir(exist_ok=True)

# Paleta kolorów czytelna także dla osób z zaburzeniami widzenia barw
KOLORY = {"swobodne": "#2a78d6", "ocena_ceny": "#eb6834"}
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a"])})

# Parametry stanowiska (takie same jak w notebooku 01)
EKRAN_SZER_PX, EKRAN_WYS_PX = 1920, 1080
EKRAN_SZER_CM, ODLEGLOSC_CM = 53.1, 60.0
PX_NA_STOPIEN = EKRAN_SZER_PX / EKRAN_SZER_CM * ODLEGLOSC_CM * np.tan(np.radians(1))
CZAS_BODZCA_MS = 10_000
ZASIEG = (0, EKRAN_SZER_PX, EKRAN_WYS_PX, 0)  # extent dla imshow: (lewo, prawo, dół, góra)

## 1. Wczytanie danych

Plik `et_fiksacje_grupa.csv` to typowy **raport fiksacji** eksportowany z oprogramowania eye-trackera (np. EyeLink Data Viewer, Tobii Pro Lab). Każdy wiersz to jedna fiksacja; czasy podano w milisekundach **od pojawienia się bodźca**.

In [ ]:
fiksacje = pd.read_csv(DANE / "et_fiksacje_grupa.csv")
obraz = plt.imread(DANE / "bodziec_plakat.png")

print(f"Liczba fiksacji: {len(fiksacje)}, liczba uczestników: {fiksacje['uczestnik'].nunique()}")
print(fiksacje.groupby("warunek")["uczestnik"].nunique().rename("liczba uczestników").to_string())
fiksacje.head()

Zanim zaczniemy rysować, warto rzucić okiem na podstawowe statystyki: liczbę fiksacji na osobę i rozkład czasów fiksacji w obu warunkach.

In [ ]:
na_osobe = fiksacje.groupby(["uczestnik", "warunek"], as_index=False).agg(
    liczba_fiksacji=("nr_fiksacji", "size"), sredni_czas_ms=("czas_trwania_ms", "mean"))

fig, osie = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=na_osobe, x="warunek", y="liczba_fiksacji", hue="warunek", palette=KOLORY, ax=osie[0])
sns.stripplot(data=na_osobe, x="warunek", y="liczba_fiksacji", color="black", alpha=0.5, ax=osie[0])
osie[0].set(title="Liczba fiksacji na osobę (10 s)", xlabel="", ylabel="Liczba fiksacji")
sns.histplot(data=fiksacje, x="czas_trwania_ms", hue="warunek", palette=KOLORY, stat="density",
             common_norm=False, element="step", bins=40, ax=osie[1])
osie[1].set(title="Rozkład czasu trwania fiksacji", xlabel="Czas fiksacji [ms]", ylabel="Gęstość")
plt.tight_layout()
plt.show()

na_osobe.groupby("warunek")[["liczba_fiksacji", "sredni_czas_ms"]].agg(["mean", "std"]).round(1)

## 2. Ścieżka wzroku (*scanpath*)

Ścieżka wzroku pokazuje **kolejność** fiksacji: koła to fiksacje (pole koła ∝ czas trwania), linie to sakady, a liczby – numer fiksacji. Pierwsza fiksacja leży zwykle w centrum ekranu, bo tam przed bodźcem wyświetlano krzyżyk fiksacyjny.

In [ ]:
KOLOR_SCIEZKI = "#2a78d6"


def rysuj_sciezke(ax, f, tytul=""):
    """Rysuje ścieżkę wzroku jednej osoby na tle bodźca."""
    ax.imshow(obraz, extent=ZASIEG)
    ax.plot(f["x_px"], f["y_px"], color="white", lw=4, alpha=0.9)  # biała „poświata” poprawia czytelność
    ax.plot(f["x_px"], f["y_px"], color=KOLOR_SCIEZKI, lw=1.5)
    ax.scatter(f["x_px"], f["y_px"], s=f["czas_trwania_ms"] * 1.5, color=KOLOR_SCIEZKI, alpha=0.8,
               edgecolor="white", linewidth=1.5, zorder=3)
    for _, fiks in f.iterrows():
        ax.text(fiks["x_px"], fiks["y_px"], str(fiks["nr_fiksacji"]), color="white", fontsize=8,
                fontweight="bold", ha="center", va="center", zorder=4)
    ax.set(xlim=(0, EKRAN_SZER_PX), ylim=(EKRAN_WYS_PX, 0), title=tytul)
    ax.set_axis_off()


fig, osie = plt.subplots(1, 2, figsize=(16, 4.8))
for os_, uczestnik in zip(osie, ["U01", "U21"]):
    f = fiksacje[fiksacje["uczestnik"] == uczestnik]
    rysuj_sciezke(os_, f, f"{uczestnik} – warunek: {f['warunek'].iloc[0]}")
plt.tight_layout()
plt.show()

Wersja **interaktywna** (Plotly): najedź kursorem na fiksację, aby zobaczyć jej numer, moment rozpoczęcia i czas trwania. ✏️ Zmień identyfikator uczestnika, aby obejrzeć inną osobę.

In [ ]:
UCZESTNIK = "U01"  # ✏️

f = fiksacje[fiksacje["uczestnik"] == UCZESTNIK]
obraz_rgb = (obraz[::2, ::2, :3] * 255).astype(np.uint8)  # zmniejszamy obraz 2× – szybsze wyświetlanie

fig = go.Figure(go.Image(z=obraz_rgb, dx=2, dy=2, hoverinfo="skip"))
fig.add_trace(go.Scatter(
    x=f["x_px"], y=f["y_px"], mode="lines+markers+text", text=f["nr_fiksacji"],
    textfont=dict(color="white", size=10),
    line=dict(color=KOLOR_SCIEZKI, width=2),
    marker=dict(size=np.sqrt(f["czas_trwania_ms"]) * 1.6, color=KOLOR_SCIEZKI, opacity=0.85,
                line=dict(color="white", width=1.5)),
    customdata=f[["start_ms", "czas_trwania_ms"]],
    hovertemplate="Fiksacja nr %{text}<br>start: %{customdata[0]} ms<br>"
                  "czas trwania: %{customdata[1]} ms<extra></extra>",
))
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_layout(title=f"Ścieżka wzroku – {UCZESTNIK}", width=960, height=580, margin=dict(l=10, r=10, t=50, b=10))
fig.show()

## 3. Mapy cieplne (KDE)

Ścieżki wzroku dobrze opisują jedną osobę, ale przy 40 osobach stają się nieczytelne. Wtedy sięgamy po **mapę cieplną**, która pokazuje, *gdzie* skupiała się uwaga całej grupy.

Mapa cieplna to **estymacja jądrowa gęstości** (*Kernel Density Estimation*, KDE): w miejscu każdej fiksacji umieszczamy „górkę” – dwuwymiarową funkcję Gaussa – i sumujemy górki ze wszystkich fiksacji:

$$\text{mapa}(x, y) = \sum_{i} w_i \cdot \exp\left(-\frac{(x - x_i)^2 + (y - y_i)^2}{2\sigma^2}\right)$$

* $w_i$ – waga fiksacji; zwykle jej **czas trwania** (dłuższe patrzenie = „cieplej”),
* $\sigma$ – szerokość jądra. Dobieramy ją do fizjologii: ok. **1° kąta widzenia** odpowiada obszarowi najostrzejszego widzenia (dołek środkowy) i typowej dokładności eye-trackera.

Jądro Gaussa jest *separowalne* (iloczyn funkcji od $x$ i od $y$), więc całą mapę policzymy jednym mnożeniem macierzy.

In [ ]:
def mapa_cieplna(x, y, wagi=None, sigma_px=PX_NA_STOPIEN, krok=4):
    """Mapa cieplna (KDE z izotropowym jądrem Gaussa) na siatce pikseli ekranu.

    Zwraca tablicę o wymiarach (wysokość / krok, szerokość / krok), która sumuje się do 1.
    """
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    wagi = np.ones(len(x)) if wagi is None else np.asarray(wagi, dtype=float)
    siatka_x = np.arange(0, EKRAN_SZER_PX, krok) + krok / 2
    siatka_y = np.arange(0, EKRAN_WYS_PX, krok) + krok / 2
    jadro_x = np.exp(-0.5 * ((siatka_x[None, :] - x[:, None]) / sigma_px) ** 2)  # (fiksacje, kolumny)
    jadro_y = np.exp(-0.5 * ((siatka_y[None, :] - y[:, None]) / sigma_px) ** 2)  # (fiksacje, wiersze)
    mapa = (jadro_y * wagi[:, None]).T @ jadro_x  # suma „górek” wszystkich fiksacji
    return mapa / mapa.sum()


# Bodziec w odcieniach szarości (rozjaśniony) – dzięki temu kolor niesie wyłącznie informację o uwadze
obraz_szary = 0.45 + 0.55 * (obraz[..., :3] @ np.array([0.299, 0.587, 0.114]))


def nakladka_rgba(mapa, cmap, granica, alfa_max=0.9):
    """Zamienia mapę na obraz RGBA: kolor z palety, a nieprzezroczystość rośnie wraz z |wartością|."""
    sila = np.clip(np.abs(mapa) / granica, 0, 1)
    if mapa.min() >= 0:  # mapa sekwencyjna: od jasnego do ciemnego odcienia jednego koloru
        rgba = plt.get_cmap(cmap)(0.3 + 0.7 * sila)
    else:  # mapa dywergentna: dwa kolory i neutralny środek
        rgba = plt.get_cmap(cmap)(0.5 + 0.5 * np.clip(mapa / granica, -1, 1))
    rgba[..., 3] = alfa_max * np.clip((sila - 0.02) / 0.98, 0, 1) ** 0.5
    return rgba


def pokaz_tlo(ax):
    """Rysuje szary bodziec jako tło dla map."""
    ax.imshow(obraz_szary, cmap="gray", vmin=0, vmax=1, extent=ZASIEG)
    ax.set_axis_off()


def pokaz_mape(ax, mapa, tytul="", cmap="Reds"):
    """Nakłada mapę cieplną na bodziec (słabe wartości są niemal przezroczyste)."""
    pokaz_tlo(ax)
    ax.imshow(nakladka_rgba(mapa, cmap, granica=mapa.max()), extent=ZASIEG)
    ax.set_title(tytul)


mapa_wszyscy = mapa_cieplna(fiksacje["x_px"], fiksacje["y_px"], wagi=fiksacje["czas_trwania_ms"])

fig, ax = plt.subplots(figsize=(12, 6.75))
pokaz_mape(ax, mapa_wszyscy, "Mapa cieplna – wszyscy uczestnicy (KDE, σ = 1°, wagi = czas fiksacji)")
plt.show()

Kilka decyzji projektowych, które warto świadomie powtarzać:

* bodziec pokazujemy **w odcieniach szarości**, żeby kolory plakatu nie myliły się z kolorami mapy,
* używamy **sekwencyjnej palety jednego koloru** (jasna → ciemna czerwień), a nieprzezroczystość nakładki rośnie z gęstością – ciemniej i „mocniej” znaczy więcej uwagi,
* unikamy popularnej w programach do eye-trackingu palety tęczowej (*jet*): wygląda efektownie, ale zniekształca odbiór – oko widzi ostre granice tam, gdzie dane zmieniają się płynnie.

### Jak szerokość jądra zmienia mapę?

Szerokość jądra $\sigma$ to najważniejszy parametr mapy cieplnej. Zbyt mała daje „plamki” wokół pojedynczych fiksacji, zbyt duża – jedną rozmytą chmurę.

In [ ]:
fig, osie = plt.subplots(2, 2, figsize=(14, 8))
for os_, sigma_deg in zip(osie.flat, [0.5, 1, 2, 4]):
    mapa = mapa_cieplna(fiksacje["x_px"], fiksacje["y_px"], wagi=fiksacje["czas_trwania_ms"],
                        sigma_px=sigma_deg * PX_NA_STOPIEN)
    pokaz_mape(os_, mapa, f"σ = {sigma_deg}° ({sigma_deg * PX_NA_STOPIEN:.0f} px)")
plt.tight_layout()
plt.show()

### Gotowe narzędzia: `scipy.stats.gaussian_kde` i `seaborn.kdeplot`

Biblioteki statystyczne potrafią dobrać szerokość jądra **automatycznie** (np. regułą Scotta). Taka reguła nie wie jednak nic o oku i ekranie – opiera się tylko na rozrzucie danych. Porównajmy.

In [ ]:
kde = gaussian_kde(np.vstack([fiksacje["x_px"], fiksacje["y_px"]]), weights=fiksacje["czas_trwania_ms"])
sigma_auto = np.sqrt(np.diag(kde.covariance)) / PX_NA_STOPIEN
print(f"Automatyczna szerokość jądra (reguła Scotta): σx = {sigma_auto[0]:.1f}°, σy = {sigma_auto[1]:.1f}°")

krok = 8
siatka_x, siatka_y = np.meshgrid(np.arange(0, EKRAN_SZER_PX, krok) + krok / 2,
                                 np.arange(0, EKRAN_WYS_PX, krok) + krok / 2)
mapa_scipy = kde(np.vstack([siatka_x.ravel(), siatka_y.ravel()])).reshape(siatka_x.shape)

fig, osie = plt.subplots(1, 2, figsize=(16, 4.8))
pokaz_mape(osie[0], mapa_scipy, "scipy.stats.gaussian_kde (szerokość automatyczna)")
pokaz_tlo(osie[1])
sns.kdeplot(data=fiksacje, x="x_px", y="y_px", weights="czas_trwania_ms", fill=False, levels=6,
            thresh=0.15, bw_adjust=0.3, color="#a50f15", linewidths=1.5, ax=osie[1])
osie[1].set(xlim=(0, EKRAN_SZER_PX), ylim=(EKRAN_WYS_PX, 0),
            title="seaborn.kdeplot (bw_adjust = 0.3) – poziomice gęstości")
plt.tight_layout()
plt.show()

Automatyczna szerokość jądra jest wyraźnie większa niż 1° (i różna w poziomie i w pionie) – mapa zlewa sąsiednie elementy plakatu w szerokie plamy. Wniosek: **parametry mapy cieplnej ustalamy świadomie** i podajemy je w opisie rycin (σ w stopniach, wagi, próg przezroczystości).

### Porównanie warunków

Mapy cieplne dla dwóch grup i ich **różnica** (paleta dywergentna: dwa kolory i neutralny środek). Normalizujemy każdą mapę do sumy 1, więc różnica pokazuje zmianę *rozkładu* uwagi, a nie liczby fiksacji.

In [ ]:
mapy = {
    warunek: mapa_cieplna(grupa["x_px"], grupa["y_px"], wagi=grupa["czas_trwania_ms"])
    for warunek, grupa in fiksacje.groupby("warunek")
}
roznica = mapy["ocena_ceny"] - mapy["swobodne"]
granica = np.abs(roznica).max()

fig, osie = plt.subplots(1, 3, figsize=(20, 4.6), layout="constrained")
pokaz_mape(osie[0], mapy["swobodne"], "Swobodne oglądanie")
pokaz_mape(osie[1], mapy["ocena_ceny"], "Zadanie: ocena ceny")
pokaz_tlo(osie[2])
osie[2].imshow(nakladka_rgba(roznica, "PuOr_r", granica), extent=ZASIEG)
osie[2].set_title("Różnica: ocena ceny − swobodne")
skala = plt.cm.ScalarMappable(cmap="PuOr_r", norm=plt.Normalize(-granica, granica))
pasek = fig.colorbar(skala, ax=osie[2], shrink=0.8, ticks=[-granica, 0, granica])
pasek.ax.set_yticklabels(["więcej: swobodne", "bez różnicy", "więcej: ocena ceny"])
plt.show()

## 4. Analiza obszarów zainteresowania (AOI)

Mapy cieplne są poglądowe, ale do **testów statystycznych** potrzebujemy liczb. Dzielimy bodziec na obszary zainteresowania (*Areas of Interest*, AOI) i dla każdej osoby liczymy miary uwagi w każdym obszarze. Tu AOI to prostokąty `(x0, y0, x1, y1)` w pikselach ekranu – w praktyce definiujemy je **przed** zebraniem danych, żeby uniknąć „dopasowywania” obszarów do wyników.

In [ ]:
AOI = {
    "Logo": (80, 60, 300, 320),
    "Hasło": (560, 95, 1800, 235),
    "Twarz": (230, 400, 630, 830),
    "Produkt": (1000, 340, 1420, 830),
    "Cena": (1515, 435, 1865, 785),
    "Przycisk": (760, 890, 1160, 1010),
}


def rysuj_aoi(ax):
    """Rysuje prostokąty AOI z podpisami."""
    for nazwa, (x0, y0, x1, y1) in AOI.items():
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="black", lw=2, ls="--"))
        ax.text(x0 + 8, y0 + 8, nazwa, va="top", fontsize=11, fontweight="bold",
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=2))


fig, ax = plt.subplots(figsize=(12, 6.75))
ax.imshow(obraz, extent=ZASIEG)
rysuj_aoi(ax)
ax.set_title("Obszary zainteresowania (AOI)")
ax.set_axis_off()
plt.show()

### Przypisanie fiksacji do AOI

Fiksacja należy do AOI, jeśli jej środek leży wewnątrz prostokąta. Fiksacje poza wszystkimi obszarami oznaczamy jako `tło`. Funkcja `np.select` sprawdza warunki po kolei i zwraca pierwszy spełniony – przy obszarach zagnieżdżonych (np. „Oczy” wewnątrz „Twarzy”) kolejność ma więc znaczenie.

In [ ]:
def przypisz_aoi(x, y):
    """Zwraca nazwę AOI dla każdej fiksacji (lub „tło”)."""
    x, y = np.asarray(x), np.asarray(y)
    warunki = [(x >= x0) & (x <= x1) & (y >= y0) & (y <= y1) for (x0, y0, x1, y1) in AOI.values()]
    return np.select(warunki, list(AOI), default="tło")


fiksacje["aoi"] = przypisz_aoi(fiksacje["x_px"], fiksacje["y_px"])
pd.crosstab(fiksacje["aoi"], fiksacje["warunek"], margins=True, margins_name="razem")

### Miary uwagi w AOI

Dla każdej osoby i każdego AOI liczymy:

| miara | definicja |
|---|---|
| **TTFF** (*Time to First Fixation*) | czas od pojawienia się bodźca do początku pierwszej fiksacji w AOI |
| **Liczba fiksacji** | ile fiksacji przypadło na AOI |
| **Całkowity czas fiksacji** | suma czasów trwania fiksacji w AOI |
| **Dwell Time** (czas przebywania) | suma czasów **wizyt** w AOI; wizyta trwa od początku pierwszej do końca ostatniej z kolejnych fiksacji w tym samym AOI, więc obejmuje też sakady wewnątrz obszaru (Holmqvist i in., 2011) |
| **Liczba wizyt** | ile razy wzrok „wchodził” do AOI |

Najpierw wyznaczamy wizyty: nowa wizyta zaczyna się, gdy AOI kolejnej fiksacji różni się od AOI poprzedniej.

In [ ]:
fiksacje = fiksacje.sort_values(["uczestnik", "nr_fiksacji"]).reset_index(drop=True)
fiksacje["nr_wizyty"] = fiksacje.groupby("uczestnik")["aoi"].transform(lambda s: (s != s.shift()).cumsum())

wizyty = fiksacje.groupby(["uczestnik", "warunek", "nr_wizyty", "aoi"], as_index=False).agg(
    start_ms=("start_ms", "min"), koniec_ms=("koniec_ms", "max"), liczba_fiksacji=("nr_fiksacji", "size"))
wizyty["czas_wizyty_ms"] = wizyty["koniec_ms"] - wizyty["start_ms"]
wizyty.head(8)

Teraz miary dla par *uczestnik × AOI*. **Ważny szczegół:** jeśli ktoś ani razu nie spojrzał na dany AOI, w danych nie ma dla niego żadnego wiersza. Musimy dopisać takie pary „ręcznie” – z liczbą fiksacji i czasem przebywania równymi **0** (a TTFF nieokreślonym, `NaN`). Inaczej średnie byłyby liczone tylko z osób, które patrzyły na obszar, i byłyby zawyżone.

In [ ]:
klucze = ["uczestnik", "warunek", "aoi"]
miary_fiksacji = fiksacje.groupby(klucze).agg(
    ttff_ms=("start_ms", "min"),
    czas_pierwszej_fiksacji_ms=("czas_trwania_ms", "first"),
    liczba_fiksacji=("nr_fiksacji", "size"),
    calkowity_czas_fiksacji_ms=("czas_trwania_ms", "sum"),
)
miary_wizyt = wizyty.groupby(klucze).agg(dwell_time_ms=("czas_wizyty_ms", "sum"),
                                         liczba_wizyt=("nr_wizyty", "size"))

# Pełna siatka: każdy uczestnik × każdy AOI (bez tła)
siatka = fiksacje[["uczestnik", "warunek"]].drop_duplicates().merge(pd.DataFrame({"aoi": list(AOI)}), how="cross")
miary = siatka.merge(miary_fiksacji.join(miary_wizyt).reset_index(), on=klucze, how="left")
zera = ["liczba_fiksacji", "calkowity_czas_fiksacji_ms", "dwell_time_ms", "liczba_wizyt"]
miary[zera] = miary[zera].fillna(0)
miary["dwell_time_proc"] = 100 * miary["dwell_time_ms"] / CZAS_BODZCA_MS

print(f"Tabela miar: {len(miary)} wierszy (= {miary['uczestnik'].nunique()} osób × {len(AOI)} AOI)")
miary.head(12).round(1)

### Podsumowanie grupowe

Średnia ± błąd standardowy średniej (SEM) w każdym warunku. Dla TTFF podajemy też **odsetek osób, które w ogóle spojrzały** na AOI (*hit ratio*) – średni TTFF liczony jest wyłącznie z tych osób.

In [ ]:
podsumowanie = miary.groupby(["aoi", "warunek"]).agg(
    odsetek_osob=("ttff_ms", lambda s: 100 * s.notna().mean()),
    ttff_srednia=("ttff_ms", "mean"), ttff_sem=("ttff_ms", "sem"),
    dwell_srednia=("dwell_time_ms", "mean"), dwell_sem=("dwell_time_ms", "sem"),
    liczba_fiksacji=("liczba_fiksacji", "mean"),
).reindex(list(AOI), level="aoi")
podsumowanie.round(0)

In [ ]:
fig, osie = plt.subplots(1, 2, figsize=(15, 4.8))
for os_, kolumna, opis in zip(osie, ["ttff_ms", "dwell_time_ms"],
                              ["Czas do pierwszej fiksacji – TTFF [ms]", "Dwell Time [ms]"]):
    sns.barplot(data=miary, x="aoi", y=kolumna, hue="warunek", order=list(AOI), palette=KOLORY,
                errorbar="se", capsize=0.1, alpha=0.85, legend=False, ax=os_)
    sns.stripplot(data=miary, x="aoi", y=kolumna, hue="warunek", order=list(AOI), palette=KOLORY,
                  dodge=True, size=3, alpha=0.5, edgecolor="black", linewidth=0.5, legend=False, ax=os_)
    os_.set(xlabel="", ylabel=opis, title=opis + " (średnia ± SEM, punkty = osoby)")
osie[0].legend(handles=[Patch(color=kolor, label=warunek) for warunek, kolor in KOLORY.items()],
               title="Warunek", loc="upper left")
plt.tight_layout()
plt.show()

Widać wyraźny **efekt zadania**: osoby oceniające cenę szybciej trafiały wzrokiem na cenę (krótszy TTFF) i dłużej się w nią wpatrywały, kosztem twarzy i hasła. Przy swobodnym oglądaniu wzrok najszybciej przyciągała twarz.

### Test statystyczny

Sprawdźmy formalnie, czy Dwell Time dla AOI „Cena” różni się między grupami. Grupy są niezależne (różne osoby), więc używamy **testu t Welcha** (nie zakłada równych wariancji) i – dla porównania – nieparametrycznego **testu U Manna–Whitneya**. Wielkość efektu opisujemy współczynnikiem **d Cohena**.

In [ ]:
def porownaj_grupy(miary, aoi, kolumna):
    """Porównuje dwie grupy dla wybranego AOI i miary; zwraca słownik z wynikami."""
    dane = miary[miary["aoi"] == aoi]
    a = dane.loc[dane["warunek"] == "swobodne", kolumna].dropna()
    b = dane.loc[dane["warunek"] == "ocena_ceny", kolumna].dropna()
    t, p_t = stats.ttest_ind(b, a, equal_var=False)
    _, p_u = stats.mannwhitneyu(b, a)
    d = (b.mean() - a.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return {"AOI": aoi, "miara": kolumna, "n swobodne": len(a), "n ocena_ceny": len(b),
            "średnia swobodne": a.mean(), "średnia ocena_ceny": b.mean(),
            "t Welcha": t, "p (t)": p_t, "p (U Manna–Whitneya)": p_u, "d Cohena": d}


wyniki_testow = pd.DataFrame([porownaj_grupy(miary, aoi, miara)
                              for aoi in ["Cena", "Twarz"] for miara in ["dwell_time_ms", "ttff_ms"]])
wyniki_testow.style.format({"średnia swobodne": "{:.0f}", "średnia ocena_ceny": "{:.0f}", "t Welcha": "{:.2f}",
                            "p (t)": "{:.2g}", "p (U Manna–Whitneya)": "{:.2g}", "d Cohena": "{:.2f}"})

Porównaj wartości *p* i *d* dla różnych miar: nie każda różnica widoczna na wykresie jest równie silna, a TTFF (z brakującymi wartościami i dużą zmiennością) daje zwykle słabsze efekty niż Dwell Time.

Dwa ważne zastrzeżenia:

* **Porównania wielokrotne.** Testując wiele AOI i wiele miar, zwiększamy ryzyko fałszywie pozytywnego wyniku – należy skorygować poziom istotności (np. metodą Bonferroniego lub Holma) albo z góry ograniczyć się do hipotez zapisanych w preregistracji.
* **Brakujące TTFF.** Osoby, które nie spojrzały na AOI, wypadają z analizy TTFF (sprawdź kolumny `n`). Jeśli takich osób jest dużo, porównanie średnich TTFF bywa mylące – warto raportować też odsetek osób i rozważyć metody analizy przeżycia (*survival analysis*).

## 5. Zapis wyników

In [ ]:
miary.round(2).to_csv(WYNIKI / "02_miary_aoi.csv", index=False)
fiksacje.to_csv(WYNIKI / "02_fiksacje_z_aoi.csv", index=False)
print("Zapisano:", *sorted(p.name for p in WYNIKI.glob("02_*")), sep="\n  ")

## Podsumowanie

* **Ścieżka wzroku** pokazuje kolejność oglądania u jednej osoby; dla grup jest nieczytelna.
* **Mapa cieplna** to estymacja jądrowa gęstości fiksacji. Jej wygląd zależy od szerokości jądra (σ ≈ 1°), wag (czas fiksacji) i palety kolorów – parametry te zawsze raportujemy.
* **Analiza AOI** zamienia obraz na liczby: TTFF, Dwell Time, liczbę fiksacji i wizyt. Pamiętamy o dopisaniu zer dla nieoglądanych obszarów i o osobach bez TTFF.
* Zadanie wykonywane przez obserwatora zmienia rozkład uwagi (efekt Yarbusa) – widać to zarówno na mapach różnicowych, jak i w testach statystycznych.

## Zadania do samodzielnego wykonania

Pięć zadań, w których powtórzysz najważniejsze kroki z tego notebooka. **Nie wymagają nowych funkcji ani pojęć** – wszystko, czego potrzebujesz, pojawiło się wyżej (przy każdym zadaniu podajemy, którą sekcję powtarza). Każde zadanie kończy się **jednym wynikiem**: liczbą wypisaną w podanym formacie albo jedną figurą.

**Zasady pracy**

1. Zanim zaczniesz, uruchom cały notebook (**Run All**) – zadania korzystają ze zmiennych i funkcji zdefiniowanych wyżej.
2. Rozwiązanie wpisz w komórce kodu pod treścią zadania.
3. **Nie nadpisuj zmiennych z notebooka** (np. `fiksacje`, `miary`, `AOI`). Nowym zmiennym nadawaj nazwy z przedrostkiem zadania: `z1_`, `z2_`, … Dzięki temu zadania nie psują sobie nawzajem danych i nie trzeba uruchamiać notebooka od nowa.
4. **Kontrola pośrednia** to wartość, którą możesz sprawdzić po drodze. Jeśli się nie zgadza, błąd jest wcześniej – popraw go, zanim pójdziesz dalej.
5. **✅ Wynik** to jedyna rzecz do oddania: liczba wypisana dokładnie w podanym formacie (z podaną liczbą miejsc po przecinku) albo figura z podanym tytułem.
6. **Do przemyślenia** – pytanie, które pomaga zrozumieć wynik. Nie trzeba go oddawać, ale warto na nie odpowiedzieć choćby w myślach.

### Zadanie 1. Pierwsze dwie sekundy – ścieżki wzroku

*Powtarzasz: sekcja 2 (ścieżka wzroku).*

**Cel:** przećwiczyć wybieranie fiksacji jednej osoby z raportu grupowego i rysowanie ścieżki wzroku.

**Krok po kroku:**

1. Z tabeli `fiksacje` wybierz fiksacje osoby `U14` (warunek `swobodne`), które **zaczęły się** w ciągu pierwszych 2 s oglądania (`start_ms < 2000`). Połącz oba warunki operatorem `&` (każdy warunek w nawiasie).
2. To samo zrób dla osoby `U38` (warunek `ocena_ceny`).
3. Utwórz figurę z dwoma panelami obok siebie: `fig, osie = plt.subplots(1, 2, figsize=(16, 4.8))` i narysuj w nich obie ścieżki funkcją `rysuj_sciezke` (sekcja 2).
4. Tytuł każdego panelu: identyfikator osoby, warunek i liczba fiksacji, np. `"U14 – swobodne – 9 fiksacji w 0–2 s"`. Tytuł całej figury (`fig.suptitle`) zacznij od „Zadanie 1”.

**Kontrola pośrednia:** osoba `U14` ma w tym przedziale 9 fiksacji.

**✅ Wynik:** jedna figura z dwiema ścieżkami wzroku (`U14` i `U38`, pierwsze 2 s).

**Do przemyślenia:** Na co każda z osób spojrzała zaraz po opuszczeniu środka ekranu? Czy już w pierwszych 2 s widać wpływ zadania (efekt Yarbusa)?

In [ ]:
# Zadanie 1 – Twoje rozwiązanie



# Wynik: jedna figura z tytułem zaczynającym się od „Zadanie 1”

### Zadanie 2. Mapy cieplne: początek i reszta oglądania

*Powtarzasz: sekcja 3 (mapy cieplne KDE).*

**Cel:** przećwiczyć liczenie i rysowanie map cieplnych dla wybranego podzbioru fiksacji.

**Krok po kroku:**

1. Podziel fiksacje wszystkich osób (oba warunki razem) na dwie grupy: rozpoczęte w pierwszych 2 s (`start_ms < 2000`) i później (`start_ms >= 2000`).
2. Dla każdej grupy policz mapę cieplną funkcją `mapa_cieplna` z ustawieniami jak w notebooku: σ = 1° (wartość domyślna) i wagi = czas trwania fiksacji (`wagi=grupa["czas_trwania_ms"]`).
3. Narysuj obie mapy obok siebie funkcją `pokaz_mape`: `fig, osie = plt.subplots(1, 2, figsize=(16, 4.8))`.
4. W tytułach paneli podaj przedział czasu i liczbę fiksacji, np. `"0–2 s (n = … fiksacji)"`. Tytuł całej figury zacznij od „Zadanie 2”.

**Kontrola pośrednia:** w pierwszych 2 s zaczęło się 288 fiksacji, a później – 1088.

**✅ Wynik:** jedna figura z dwiema mapami cieplnymi (0–2 s i 2–10 s).

**Do przemyślenia:** Gdzie skupia się uwaga w pierwszych 2 s, a gdzie później? Funkcja `mapa_cieplna` normalizuje mapę do sumy 1 – dlaczego jest to ważne, skoro w obu grupach jest bardzo różna liczba fiksacji?

In [ ]:
# Zadanie 2 – Twoje rozwiązanie



# Wynik: jedna figura z tytułem zaczynającym się od „Zadanie 2”

### Zadanie 3. Dlaczego trzeba dopisywać zera?

*Powtarzasz: sekcja 4 („Miary uwagi w AOI” i „Podsumowanie grupowe”).*

**Cel:** zobaczyć na liczbach, jak bardzo zawyżylibyśmy średni Dwell Time, gdybyśmy nie dopisali zer dla osób, które nie spojrzały na AOI.

**Krok po kroku:**

1. Z tabeli `miary` wybierz wiersze dla AOI `"Logo"` i warunku `"ocena_ceny"` (20 wierszy – po jednym na osobę).
2. Policz średni Dwell Time (`dwell_time_ms`) na dwa sposoby:
   * (a) ze wszystkich osób – tak jak w tabeli `podsumowanie`,
   * (b) tylko z osób, które na logo w ogóle spojrzały (`dwell_time_ms > 0`) – tak wyglądałby wynik, gdybyśmy zapomnieli dopisać zera.
3. Policz, ile osób spojrzało na logo.

**Kontrola pośrednia:** wariant (a) to ok. 220 ms (kolumna `dwell_srednia` w tabeli `podsumowanie`).

**✅ Wynik:** `print(f"Zadanie 3: bez zer = {z3_bez_zer:.0f} ms (n = {z3_n})")`

**Do przemyślenia:** O ile procent zawyżylibyśmy wynik? Który wariant odpowiada na pytanie „jak długo przeciętny uczestnik patrzył na logo?”, a który na pytanie „jak długo patrzyli ci, którzy je zauważyli?”.

In [ ]:
# Zadanie 3 – Twoje rozwiązanie



# print(f"Zadanie 3: bez zer = {z3_bez_zer:.0f} ms (n = {z3_n})")

### Zadanie 4. Fiksacje a wizyty

*Powtarzasz: sekcja 4 (wizyty w AOI, liczba fiksacji i liczba wizyt).*

**Cel:** zrozumieć różnicę między **fiksacją** a **wizytą** w AOI.

**Krok po kroku:**

1. Dla każdego AOI zsumuj w tabeli `miary` liczbę fiksacji (`liczba_fiksacji`) i liczbę wizyt (`liczba_wizyt`) – dla wszystkich osób i obu warunków razem. *Wskazówka:* `miary.groupby("aoi")[["liczba_fiksacji", "liczba_wizyt"]].sum()`.
2. Podziel sumę fiksacji przez sumę wizyt – otrzymasz średnią liczbę fiksacji przypadającą na jedną wizytę w danym AOI.
3. Wyświetl wynik dla wszystkich 6 AOI, posortowany malejąco (`sort_values(ascending=False)`).

**Kontrola pośrednia:** dla „Logo” wychodzi ok. 1,09 fiksacji na wizytę.

**✅ Wynik:** `print(f"Zadanie 4: Hasło = {z4_na_wizyte['Hasło']:.2f} fiksacji na wizytę")`

**Do przemyślenia:** Dlaczego dla „Hasła” wynik jest tak duży? (Zajrzyj do opisu danych w `dane/README.md` – jak oglądano hasło?) Dla którego AOI różnica między Dwell Time a całkowitym czasem fiksacji powinna być największa i dlaczego?

In [ ]:
# Zadanie 4 – Twoje rozwiązanie



# print(f"Zadanie 4: Hasło = {z4_na_wizyte['Hasło']:.2f} fiksacji na wizytę")

### Zadanie 5. Wiele testów naraz – korekta Bonferroniego

*Powtarzasz: sekcja 4 („Test statystyczny” i uwaga o porównaniach wielokrotnych).*

**Cel:** przećwiczyć porównanie grup funkcją `porownaj_grupy` i zobaczyć, jak korekta na porównania wielokrotne zmienia wnioski.

**Krok po kroku:**

1. Funkcją `porownaj_grupy` porównaj warunki dla **wszystkich 6 AOI** (`AOI`) i **dwóch miar**: `dwell_time_ms` i `ttff_ms` – razem 12 testów. Wyniki zbierz w jednej tabeli `pd.DataFrame`, tak jak `wyniki_testow` w notebooku (lista słowników → `pd.DataFrame`).
2. Policz, ile testów daje p < 0,05 (kolumna `"p (t)"` – test t Welcha).
3. Zastosuj **korektę Bonferroniego**: pomnóż każdą wartość p przez liczbę testów (12), a wyniki większe niż 1 zamień na 1 (`np.minimum(..., 1)`). Zapisz je w nowej kolumnie, np. `"p Bonferroni"`.
4. Policz, ile testów ma skorygowane p < 0,05.

**Kontrola pośrednia:** bez korekty istotnych jest 5 z 12 testów.

**✅ Wynik:** `print(f"Zadanie 5: po korekcie istotnych jest {z5_n_bonf} z 12 testów")`

**Do przemyślenia:** Które wyniki „przestały” być istotne po korekcie? Dlaczego TTFF daje zwykle słabsze efekty niż Dwell Time (zob. kolumny `n` i komentarz pod tabelą testów)?

In [ ]:
# Zadanie 5 – Twoje rozwiązanie



# print(f"Zadanie 5: po korekcie istotnych jest {z5_n_bonf} z 12 testów")

## Literatura

* Holmqvist, K., Nyström, M., Andersson, R., Dewhurst, R., Jarodzka, H., i van de Weijer, J. (2011). *Eye tracking: A comprehensive guide to methods and measures*. Oxford University Press.
* Hessels, R. S., Kemner, C., van den Boomen, C., i Hooge, I. T. C. (2016). The area-of-interest problem in eyetracking research: A noise-robust solution for face and sparse stimuli. *Behavior Research Methods, 48*(4), 1694–1712.
* Le Meur, O., i Baccino, T. (2013). Methods for comparing scanpaths and saliency maps: Strengths and weaknesses. *Behavior Research Methods, 45*(1), 251–266.
* Orquin, J. L., i Holmqvist, K. (2018). Threats to the validity of eye-movement research in psychology. *Behavior Research Methods, 50*(4), 1645–1656.
* Yarbus, A. L. (1967). *Eye movements and vision*. Plenum Press.